## **Bank Nifty Predictability Under Regulatory Expiry-Structure Reform**

### **Notebook 04: Volatility Regime Detection via Gaussian HMM**

[![hmmlearn](https://img.shields.io/badge/hmmlearn-Latest-blue)](https://hmmlearn.readthedocs.io/)

---

**Scope:** Train an unsupervised 2-state Gaussian Hidden Markov Model (HMM) on training split returns and realized volatility to label market volatility regimes, avoiding training information leakage.

---


### **Pipeline Context**

This is a sequential 9-notebook experimental workflow:

1. **01_data_collection.ipynb**
2. **02_data_cleaning_eda.ipynb**
3. **03_feature_engineering.ipynb**
4. **04_volatility_regime_detection.ipynb**
5. **05_baseline_models.ipynb**
6. **06_proposed_model.ipynb**
7. **07_ablation_study.ipynb**
8. **08_evaluation_and_diagnostics.ipynb**
9. **09_figures_for_paper.ipynb**

---
**Current Location:** 4. **04_volatility_regime_detection.ipynb** <- *You are here*

---


## 1. Introduction

### 1.1 Purpose of This Notebook
Financial markets alternate between distinct states or "regimes" (e.g. calm periods vs. high-volatility stress regimes). To capture this structure without leakage:
1. We split the data chronologically into Train (60%), Validation (20%), and Test (20%).
2. We fit a 2-state Gaussian Hidden Markov Model (HMM) on the training set's log returns and rolling realized volatility.
3. We sort the HMM states so State 0 is consistently the Calm regime (low variance) and State 1 is the Stressed regime (high variance).
4. We predict regime probabilities across all splits and export the processed datasets.

## 2. Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

project_dir = Path("e:/BN_Structure_Reforms/banknifty-expiry-volatility-ml")
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src import config
from src.regime_detection import VolatilityRegimeDetector

## 3. Dataset Splitting and target Alignment

### 3.1 Next-Day Return Shift

In [2]:
df = pd.read_parquet(config.INTERIM_DATA_DIR / "banknifty_features.parquet")
df['Date'] = pd.to_datetime(df['Date'])

# Align target: next-day return
df['Target'] = df['Return'].shift(-1)
df = df.dropna()

# Chronological partition
n = len(df)
train_end = int(n * 0.60)
val_end = int(n * 0.80)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train observations: {len(train_df)}")
print(f"Validation observations: {len(val_df)}")
print(f"Test observations: {len(test_df)}")

Train observations: 441
Validation observations: 147
Test observations: 147


## 4. Unsupervised HMM Volatility Regime Fitting

In [3]:
# Fit HMM strictly on the training set
detector = VolatilityRegimeDetector(n_components=2, random_state=config.RANDOM_SEED)
detector.fit(train_df)

print("[OK] HMM fit completed.")
print("Transition Matrix:\n", detector.model.transmat_)
print("State Means (Return, Vol):\n", detector.model.means_)
print("State Covariances:\n", detector.model.covars_)

Sorting HMM states: swapping State 0 and State 1 to make State 0 low-volatility.
[OK] HMM fit completed.
Transition Matrix:
 [[0.94881933 0.05118067]
 [0.05125069 0.94874931]]
State Means (Return, Vol):
 [[-5.08230697]
 [-4.52135305]]
State Covariances:
 [[[0.04511395]]

 [[0.09654953]]]


## 5. Projecting Regimes Across Splits

In [4]:
# Predict probabilities on all splits
train_probs = detector.predict_proba(train_df)
val_probs = detector.predict_proba(val_df)
test_probs = detector.predict_proba(test_df)

train_states = detector.predict(train_df)
val_states = detector.predict(val_df)
test_states = detector.predict(test_df)

for split_df, probs, states in zip([train_df, val_df, test_df], [train_probs, val_probs, test_probs], [train_states, val_states, test_states]):
    split_df['Regime_Prob_Calm'] = probs[:, 0]
    split_df['Regime_Prob_Stressed'] = probs[:, 1]
    split_df['Regime'] = states

## 6. Export Processed Datasets

In [5]:
joblib.dump(detector, config.MODELS_DIR / "hmm_regime_model.pkl")

train_df.to_parquet(config.PROCESSED_DATA_DIR / "train.parquet", index=False)
val_df.to_parquet(config.PROCESSED_DATA_DIR / "val.parquet", index=False)
test_df.to_parquet(config.PROCESSED_DATA_DIR / "test.parquet", index=False)
print("[OK] Saved processed splits and HMM model object.")

[OK] Saved processed splits and HMM model object.
